In [ ]:
import kagglehub
import os
import pandas as pd
import nltk
import string

from nltk.tokenize import word_tokenize
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

In [ ]:
path = kagglehub.dataset_download("sahideseker/chatbot-intent-classification-dataset")
print(path)

100%|██████████| 2.29k/2.29k [00:00<00:00, 3.33MB/s]

Extracting files...
/root/.cache/kagglehub/datasets/sahideseker/chatbot-intent-classification-dataset/versions/1


In [ ]:
print(os.listdir(path))

['chatbot_intent_classification.csv']


In [ ]:
file_path = os.path.join(path, "chatbot_intent_classification.csv")
df = pd.read_csv(file_path)
df.head()

,user_input,intent
0,"Cancel my subscription, please.",cancellation
1,I forgot my password.,password_reset
2,Can you help me with my account?,account_help
3,I want to return my order.,return_request
4,What time do you open?,business_hours


In [ ]:
df = df[["user_input", "intent"]]
df.head()

,user_input,intent
0,"Cancel my subscription, please.",cancellation
1,I forgot my password.,password_reset
2,Can you help me with my account?,account_help
3,I want to return my order.,return_request
4,What time do you open?,business_hours


In [ ]:
nltk.download('punkt')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [ ]:
def preprocess(text):
    text = str(text).lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    tokens = word_tokenize(text)
    return " ".join(tokens)

df['clean_text'] = df['user_input'].apply(preprocess)

In [ ]:
vectorizer = TfidfVectorizer()

X = vectorizer.fit_transform(df['clean_text'])
y = df['intent']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
model = MultinomialNB()
model.fit(X_train, y_train)

MultinomialNB()

In [ ]:
y_pred = model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))

Accuracy: 1.0


In [ ]:
def predict_intent(user_input):
    user_input = preprocess(user_input)
    vector = vectorizer.transform([user_input])
    return model.predict(vector)[0]

In [ ]:
print(predict_intent("hello"))
print(predict_intent("book a flight"))
print(predict_intent("cancel order"))

business_hours
business_hours
cancellation


In [ ]:
print("Chatbot Started (type 'exit' to stop)")

while True:
    user_input = input("You: ")

    if user_input.lower() == "exit":
        print("Bot: Goodbye!")
        break

    intent = predict_intent(user_input)
    print("Bot Intent:", intent)

Chatbot Started (type 'exit' to stop)
Bot Intent: order_status
